In [ ]:
import os
import random
import math
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import (
    Input, TimeDistributed, GlobalAveragePooling2D,
    LSTM, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [4]:
# =========================================================
# 0) GPU SAFETY (won't hurt CPU mode)
# =========================================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU memory growth enabled")
    except Exception as e:
        print("⚠️ Could not set GPU memory growth:", e)
else:
    print("ℹ️ No GPU detected (CPU mode)")

ℹ️ No GPU detected (CPU mode)


In [5]:
DATA_DIR = r"C:\Users\Rohit94\Documents\project_2023\DeepFakeVideo\data"
REAL_DIR = os.path.join(DATA_DIR, "Real")
FAKE_DIR = os.path.join(DATA_DIR, "Fake")

IMG_SIZE = (224, 224)

# CPU-safe defaults
SEQ_LEN = 4
BATCH_SIZE = 2

EPOCHS_STAGE1 = 5
EPOCHS_STAGE2 = 3
VAL_SPLIT = 0.2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)



In [6]:
# =========================================================
# 2) LIST IMAGES
# =========================================================
def list_images(folder):
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    if not os.path.isdir(folder):
        return []
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(exts)
    ])

real_imgs = list_images(REAL_DIR)
fake_imgs = list_images(FAKE_DIR)

print(f"Real images: {len(real_imgs)}")
print(f"Fake images: {len(fake_imgs)}")


Real images: 5413
Fake images: 5492


In [7]:
if len(real_imgs) < SEQ_LEN or len(fake_imgs) < SEQ_LEN:
    raise ValueError(f"Need at least {SEQ_LEN} images in BOTH Real and Fake folders.")

In [8]:
# =========================================================
# 3) BUILD PSEUDO-SEQUENCES
# (only way to use LSTM with flat image folders)
# =========================================================
def build_pseudo_sequences(image_paths, label, seq_len):
    paths = image_paths.copy()
    random.shuffle(paths)

    sequences = []
    for i in range(0, len(paths) - seq_len + 1, seq_len):
        seq = paths[i:i + seq_len]
        if len(seq) == seq_len:
            sequences.append((seq, label))
    return sequences

real_samples = build_pseudo_sequences(real_imgs, 0, SEQ_LEN)
fake_samples = build_pseudo_sequences(fake_imgs, 1, SEQ_LEN)

samples = real_samples + fake_samples
random.shuffle(samples)

print(f"Pseudo sequences created: {len(samples)}")
print(f"Real sequences: {len(real_samples)} | Fake sequences: {len(fake_samples)}")

if len(samples) < 10:
    raise ValueError("Too few pseudo-sequences. Add more images or reduce SEQ_LEN.")

Pseudo sequences created: 2726
Real sequences: 1353 | Fake sequences: 1373


In [9]:
# =========================================================
# 4) TRAIN / VAL SPLIT
# =========================================================
n_total = len(samples)
n_val = int(n_total * VAL_SPLIT)

val_samples = samples[:n_val]
train_samples = samples[n_val:]

print(f"Total: {n_total} | Train: {len(train_samples)} | Val: {len(val_samples)}")



Total: 2726 | Train: 2181 | Val: 545


In [10]:
# =========================================================
# 5) IMAGE LOAD + PREPROCESS
# =========================================================
def load_and_preprocess_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img

def load_sequence(frame_paths):
    imgs = tf.map_fn(load_and_preprocess_image, frame_paths, fn_output_signature=tf.float32)
    return imgs


# =========================================================
# 6) MEMORY-SAFE DATASET (GENERATOR)
# =========================================================
def gen_samples(samples_list):
    for seq_paths, label in samples_list:
        yield np.array(seq_paths, dtype=str), np.float32(label)

def make_dataset(samples_list, batch_size, training=True):
    output_signature = (
        tf.TensorSpec(shape=(SEQ_LEN,), dtype=tf.string),
        tf.TensorSpec(shape=(), dtype=tf.float32),
    )

    ds = tf.data.Dataset.from_generator(
        lambda: gen_samples(samples_list),
        output_signature=output_signature
    )

    if training:
        ds = ds.shuffle(min(len(samples_list), 1000), seed=SEED, reshuffle_each_iteration=True)

    def _map_fn(seq_paths, label):
        x = load_sequence(seq_paths)
        return x, label

    ds = ds.map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_samples, BATCH_SIZE, training=True)
val_ds = make_dataset(val_samples, BATCH_SIZE, training=False)


In [11]:
# =========================================================
# 7) FIX "Unknown" STEPS
# =========================================================
steps_per_epoch = math.ceil(len(train_samples) / BATCH_SIZE)
val_steps = math.ceil(len(val_samples) / BATCH_SIZE)

print("steps_per_epoch:", steps_per_epoch)
print("val_steps:", val_steps)


steps_per_epoch: 1091
val_steps: 273


In [12]:
# =========================================================
# 8) BUILD HYBRID MODEL
# =========================================================
def build_vgg16_lstm(seq_len):
    vgg = VGG16(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
    vgg.trainable = False

    video_input = Input(shape=(seq_len, 224, 224, 3), name="frames")

    x = TimeDistributed(vgg, name="td_vgg")(video_input)
    x = TimeDistributed(GlobalAveragePooling2D(), name="td_gap")(x)

    x = BatchNormalization()(x)
    x = LSTM(128)(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)

    output = Dense(1, activation="sigmoid")(x)

    return Model(video_input, output, name="VGG16_LSTM_PseudoSeq")

model = build_vgg16_lstm(SEQ_LEN)
model.summary()



Model: "VGG16_LSTM_PseudoSeq"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 frames (InputLayer)         [(None, 4, 224, 224, 3)]  0         
                                                                 
 td_vgg (TimeDistributed)    (None, 4, 7, 7, 512)      14714688  
                                                                 
 td_gap (TimeDistributed)    (None, 4, 512)            0         
                                                                 
 batch_normalization_2 (Batc  (None, 4, 512)           2048      
 hNormalization)                                                 
                                                                 
 lstm_2 (LSTM)               (None, 128)               328192    
                                                                 
 dropout_4 (Dropout)         (None, 128)               0         
                                              

# Build Hybrid VGG16 + LSTM Model

In [13]:
# =========================================================
# 9) COMPILE
# =========================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)


In [14]:
# =========================================================
# 10) CALLBACKS
# =========================================================
callbacks = [
    EarlyStopping(monitor="val_auc", mode="max", patience=2, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
    ModelCheckpoint("vgg16_lstm_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]


# =========================================================
# 11) TRAIN STAGE 1
# =========================================================
print("\n======== Stage 1: Train classifier head ========")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    callbacks=callbacks
)



======== Stage 1: Train classifier head ========
Epoch 1/5
1091/1091 [==============================] - 1402s 1s/step - loss: 0.6775 - accuracy: 0.5562 - auc: 0.6019 - precision: 0.5894 - recall: 0.4034 - val_loss: 0.5809 - val_accuracy: 0.7780 - val_auc: 0.8582 - val_precision: 0.7508 - val_recall: 0.8259 - lr: 1.0000e-04
Epoch 2/5
1091/1091 [==============================] - 322s 295ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - auc: 0.0000e+00 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 0.5809 - val_accuracy: 0.7780 - val_auc: 0.8582 - val_precision: 0.7508 - val_recall: 0.8259 - lr: 1.0000e-04


# Evaluation: Accuracy + Classification Report

In [15]:
# =========================================================
# 12) FINE-TUNE STAGE 2
# =========================================================
print("\n======== Stage 2: Fine-tune last VGG block ========")
vgg_backbone = model.get_layer("td_vgg").layer
vgg_backbone.trainable = True

for layer in vgg_backbone.layers:
    if "block5" in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    callbacks=callbacks
)


# =========================================================
# 13) EVALUATION (ACCURACY + REPORT + MATRIX)
# =========================================================
print("\n======== Evaluation on Validation Set ========")
y_prob = model.predict(val_ds, steps=val_steps, verbose=0).ravel()

# Collect true labels from val_ds (limited to val_steps)
y_true_batches = []
for i, (_, y) in enumerate(val_ds):
    if i >= val_steps:
        break
    y_true_batches.append(y.numpy().ravel())

y_true = np.concatenate(y_true_batches, axis=0)

# Align lengths if last batch mismatch
min_len = min(len(y_true), len(y_prob))
y_true = y_true[:min_len]
y_prob = y_prob[:min_len]

y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
print("Validation Accuracy:", acc)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Real", "Fake"]))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


======== Stage 2: Fine-tune last VGG block ========
Epoch 1/3
1091/1091 [==============================] - 1565s 1s/step - loss: 0.5836 - accuracy: 0.7093 - auc: 0.7867 - precision: 0.7142 - recall: 0.7090 - val_loss: 0.4188 - val_accuracy: 0.8734 - val_auc: 0.9481 - val_precision: 0.8764 - val_recall: 0.8667 - lr: 1.0000e-05
Epoch 2/3
1091/1091 [==============================] - 262s 240ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - auc: 0.0000e+00 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 0.4188 - val_accuracy: 0.8734 - val_auc: 0.9481 - val_precision: 0.8764 - val_recall: 0.8667 - lr: 1.0000e-05

======== Evaluation on Validation Set ========
Validation Accuracy: 0.8733944954128441

Classification Report:
              precision    recall  f1-score   support

        Real       0.87      0.88      0.88       275
        Fake       0.88      0.87      0.87       270

    accuracy                           0.87       545
   macro avg       0.87      0.87      0.87